In [ ]:
# Install Kaggle API
!pip install kaggle -q

# Create kaggle directory
import os
os.makedirs('/root/.kaggle', exist_ok=True)

In [ ]:
!pip install kaggle -q
import os
os.makedirs('/root/.kaggle', exist_ok=True)

# Create kaggle.json manually using your token


In [ ]:
!kaggle datasets download -d prathumarikeri/indian-sign-language-isl
!unzip -q indian-sign-language-isl.zip -d /content/dataset
!ls /content/dataset

In [ ]:
import os
folders = os.listdir('/content/dataset')
print(f"Found {len(folders)} class folders:")
print(sorted(folders))

In [ ]:
import os

# Check what's inside Indian folder
print(os.listdir('/content/dataset/Indian'))


In [ ]:
!pip install mediapipe -q
!pip install opencv-python -q

In [ ]:
import os, cv2, copy, itertools
import numpy as np
import pandas as pd
import mediapipe as mp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import warnings
warnings.filterwarnings('ignore')

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
mp_hands = mp.solutions.hands
hands_static = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.3,
)

def calc_landmark_list(image, landmarks):
    h, w = image.shape[:2]
    return [
        [min(int(lm.x * w), w-1), min(int(lm.y * h), h-1)]
        for lm in landmarks.landmark
    ]

def pre_process_landmark(landmark_list):
    temp = copy.deepcopy(landmark_list)
    base_x, base_y = temp[0]
    for p in temp:
        p[0] -= base_x
        p[1] -= base_y
    flat = list(itertools.chain.from_iterable(temp))
    max_val = max(map(abs, flat)) if flat else 1
    return [v / max_val for v in flat] if max_val != 0 else flat

def extract_landmarks(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = hands_static.process(img_rgb)
    if not result.multi_hand_landmarks:
        img_bright = cv2.convertScaleAbs(img, alpha=1.3, beta=30)
        result = hands_static.process(cv2.cvtColor(img_bright, cv2.COLOR_BGR2RGB))
        if not result.multi_hand_landmarks:
            return None
    lm = calc_landmark_list(img_rgb, result.multi_hand_landmarks[0])
    return pre_process_landmark(lm)

print("✅ Functions ready")

In [ ]:
# if mediapipe version error
!pip install mediapipe==0.10.13 protobuf==4.25.8 -q --no-deps

In [ ]:
import mediapipe as mp
print(mp.__version__)
print(hasattr(mp, 'solutions'))

In [ ]:
import mediapipe as mp
import numpy as np
import cv2
import copy
import itertools

# ── Fix for mediapipe 0.10.x ──────────────────────────
from mediapipe.python.solutions import hands as mp_hands_module
from mediapipe.python.solutions import drawing_utils as mp_drawing

mp_hands = mp_hands_module

hands_static = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.3,
)

print("✅ MediaPipe hands loaded!")

def calc_landmark_list(image, landmarks):
    h, w = image.shape[:2]
    return [
        [min(int(lm.x * w), w - 1), min(int(lm.y * h), h - 1)]
        for lm in landmarks.landmark
    ]

def pre_process_landmark(landmark_list):
    temp = copy.deepcopy(landmark_list)
    base_x, base_y = temp[0]
    for p in temp:
        p[0] -= base_x
        p[1] -= base_y
    flat = list(itertools.chain.from_iterable(temp))
    max_val = max(map(abs, flat)) if flat else 1
    return [v / max_val for v in flat] if max_val != 0 else flat

def extract_landmarks(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    result = hands_static.process(img_rgb)
    if not result.multi_hand_landmarks:
        img_bright = cv2.convertScaleAbs(img, alpha=1.3, beta=30)
        img_rgb2 = cv2.cvtColor(img_bright, cv2.COLOR_BGR2RGB)
        result = hands_static.process(img_rgb2)
        if not result.multi_hand_landmarks:
            return None
    lm = calc_landmark_list(img_rgb, result.multi_hand_landmarks[0])
    return pre_process_landmark(lm)

print("✅ Functions ready!")

In [ ]:
import os
import string

# Class order — MUST match detection_server.py exactly
CLASSES = ["1","2","3","4","5","6","7","8","9"] + list(string.ascii_uppercase)
print(f"Total classes: {len(CLASSES)}")  # Should be 35

DATASET_PATH = '/content/dataset/Indian'

X = []
y = []
failed = 0
total = 0

print("Extracting landmarks from images...")
print("This will take 10-15 minutes on CPU, 3-5 mins on GPU...")

for class_name in CLASSES:
    class_dir = os.path.join(DATASET_PATH, class_name)
    if not os.path.exists(class_dir):
        print(f"  ⚠️  Missing folder: {class_name}")
        continue

    images = [f for f in os.listdir(class_dir)
              if f.lower().endswith(('.jpg','.jpeg','.png'))]

    # Limit to 500 per class to save time
    images = images[:500]

    class_success = 0
    for img_file in images:
        total += 1
        path = os.path.join(class_dir, img_file)
        landmarks = extract_landmarks(path)
        if landmarks is not None and len(landmarks) == 42:
            X.append(landmarks)
            y.append(class_name)
            class_success += 1
        else:
            failed += 1

    print(f"  {class_name}: {class_success}/{len(images)} ✅")

print(f"\n✅ Done! Total: {len(X)} samples, {failed} failed")

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

X = np.array(X, dtype=np.float32)
y = np.array(y)

# Encode labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(CLASSES)
y_encoded = le.transform(y)

print(f"X shape: {X.shape}")
print(f"Unique classes: {len(np.unique(y_encoded))}")

# ── Augmentation ──────────────────────────────────────
def augment_landmarks(landmarks, n=4):
    augmented = []
    lm = np.array(landmarks)
    for _ in range(n):
        aug = lm.copy()
        aug += np.random.normal(0, 0.01, aug.shape)  # noise
        aug *= np.random.uniform(0.92, 1.08)          # scale
        aug = np.clip(aug, -1.5, 1.5)
        augmented.append(aug)
    return augmented

print("Augmenting data (4x)...")
X_aug = list(X)
y_aug = list(y_encoded)

for sample, label in zip(X, y_encoded):
    augs = augment_landmarks(sample, n=4)
    X_aug.extend(augs)
    y_aug.extend([label] * len(augs))

X_aug = np.array(X_aug, dtype=np.float32)
y_aug = np.array(y_aug)

print(f"After augmentation: {len(X_aug)} samples")

# ── Split ──────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_aug, y_aug, test_size=0.2, random_state=42, stratify=y_aug
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
print("✅ Data ready!")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

# ── Build Model ───────────────────────────────────────
def build_model(input_dim=42, num_classes=35):
    inputs = layers.Input(shape=(input_dim,))

    # Block 1
    x = layers.Dense(512)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    # Block 2
    x = layers.Dense(512)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.3)(x)

    # Block 3 — residual
    shortcut = layers.Dense(256)(x)
    x = layers.Dense(256)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Add()([x, shortcut])

    # Block 4
    x = layers.Dense(256)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.25)(x)

    # Block 5
    x = layers.Dense(128)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.2)(x)

    # Block 6
    x = layers.Dense(128)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Output
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model

model = build_model()
model.summary()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ─────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        'best_model.h5',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
]

# ── Train ─────────────────────────────────────────────
print("\n🚀 Training started...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=150,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training complete!")
print(f"Best val accuracy: {max(history.history['val_accuracy'])*100:.2f}%")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# ── Test Evaluation ───────────────────────────────────
print("Evaluating on test set...")
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n🎯 Test Accuracy: {test_acc*100:.2f}%")

# ── Per class report ──────────────────────────────────
y_pred = np.argmax(model.predict(X_test), axis=1)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=CLASSES))

# ── Confusion matrix ──────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'Confusion Matrix — Test Accuracy: {test_acc*100:.2f}%')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100)
plt.show()

# ── Save model ────────────────────────────────────────
model.save('model.h5')
print("\n✅ model.h5 saved!")

# ── Download ──────────────────────────────────────────
from google.colab import files
files.download('model.h5')
files.download('confusion_matrix.png')
print("✅ Files downloading...")